In [10]:
import os
import json
import glob
from shapely.geometry import shape, mapping
from shapely.affinity import scale
from pathlib import Path

# --- CONFIGURACIÓN ---
RAW_DATA_DIR = os.path.join("C:/Users/paez/Documents/Doct_2026/yolo26_rats")
WSI_IMAGES_DIR = os.path.join(RAW_DATA_DIR, "data/raw/wsi_images")
ANNOTATIONS_DIR = os.path.join(RAW_DATA_DIR, "data/raw/annotations")
OUTPUT_DIR = os.path.join(RAW_DATA_DIR,"data/raw/annotations_resized")
METADATA_DIR = os.path.join(RAW_DATA_DIR,"data/raw/metadata")
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:

# Valores obtenidos de tu meta.json y las dimensiones reales


print(f"Factor de escala detectado: {DOWNSAMPLE_FACTOR} (Reduciendo a 1/4)")

def scale_geojson_files():
    metajson_files = sorted(glob.glob(os.path.join(METADATA_DIR, "*.meta.json")))

    geojson_files = sorted(glob.glob(os.path.join(ANNOTATIONS_DIR, "*.geojson")))
    for file_path_m,file_path_g in  zip(metajson_files,geojson_files):

        file_name = os.path.basename(file_path_g)
        print(f"Procesando: {file_name}...")
        
        with open(file_path_g, 'r') as f:
            data_geojson = json.load(f)
        
        with open(file_path_m, 'r') as f:
            data_meta = json.load(f)
        
        ORIGINAL_WIDTH = data_meta['original_width']
        CURRENT_WIDTH = data_meta['current_width']
        # Calculamos el factor: 8686 / 34745 ≈ 0.25
        DOWNSAMPLE_FACTOR = CURRENT_WIDTH / ORIGINAL_WIDTH
        print(f"Factor de escala detectado: {DOWNSAMPLE_FACTOR} (Reduciendo a 1/4)")
        # Procesar cada feature (anotación)
        if 'features' in data_geojson:
            for feature in data_geojson['features']:
                # 1. Convertir a objeto Shapely
                geom = shape(feature['geometry'])
                
                # 2. ESCALAR: origin=(0,0) es clave para que no se desplacen
                scaled_geom = scale(geom, xfact=DOWNSAMPLE_FACTOR, yfact=DOWNSAMPLE_FACTOR, origin=(0,0))
                
                # 3. Guardar de nuevo en el JSON
                feature['geometry'] = mapping(scaled_geom)
        
        # Visualizar escalado - imagen + anotación
        
        # Guardar el nuevo archivo corregido
        #output_path = os.path.join(OUTPUT_DIR, file_name)
        #with open(output_path, 'w') as f:
        #    json.dump(data, f)

    print(f"\n✅ ¡Hecho! Anotaciones guardadas en: {OUTPUT_DIR}")

# Ejecutar
scale_geojson_files()

In [20]:
metajson_files = sorted(glob.glob(os.path.join(METADATA_DIR, "*.meta.json")))

geojson_files = sorted(glob.glob(os.path.join(ANNOTATIONS_DIR, "*.geojson")))
for m in  metajson_files[:1]:

    #file_name_g = os.path.basename(g)
    file_name_m = os.path.basename(m)
    with open(m, 'r') as f:
            data_meta = json.load(f)
    print(data_meta)   
    ORIGINAL_WIDTH = data_meta['width_px']
    ORIGINAL_HEIGHT = data_meta['height_px']
    
    # Calculamos el factor: 8686 / 34745 ≈ 0.25
    #DOWNSAMPLE_FACTOR = CURRENT_WIDTH / ORIGINAL_WIDTH
    print(ORIGINAL_WIDTH)
    print(ORIGINAL_HEIGHT)
    #print(CURRENT_WIDTH)
    #print(f"Factor de escala detectado: {DOWNSAMPLE_FACTOR} (Reduciendo a 1/4)")

{'imageName': '2023_03_16__11_24__17639.czi - Scene #01', 'exportBase': '2023_03_16__11_24__17639.czi - Scene #01', 'width_px': 34745, 'height_px': 24958, 'pixel_width_um': 0.2202474480078368, 'pixel_height_um': 0.2202474480078368, 'pixel_unit': 'µm', 'note': 'GeoJSON geometries are in IMAGE PIXELS. Use pixel size above if you need microns.'}
34745
24958
